In [46]:
import json
from transformers import AutoProcessor
import sys 
import os 
# current_file_path = os.path.dirname(os.path.abspath(__file__))
# module_path = os.path.join(current_file_path, "../")
# sys.path.append(module_path)
# from models.qwen2_5_vl import Qwen2VLRetForConditionalGeneration
import torch 
import argparse
from dataset.datasets_mbeir import QueryDataset, CandidateDataset
from collators.mbeir_eval import MbeirQueryDataCollator, MbeirCandidateDataCollator
from torch.utils.data import DataLoader 
import torch.nn.functional as F 
from accelerate import Accelerator
import accelerate
import numpy as np
DATASET_QUERY_NUM_UPPER_BOUND = 500000
DATASET_CAN_NUM_UPPER_BOUND = 10000000

NUM_QUERIES = 30
NUM_CANDIDATES_FOR_POOL = 300 # Number of candidates to use in the evaluation pool for simulated retrieval
MAX_RETRIEVED_PER_QUERY = 50 # For simulated retrieval and recall calculation up to K=50


In [3]:

def unhash_qid(hashed_qid):
    dataset_id = hashed_qid // DATASET_QUERY_NUM_UPPER_BOUND
    data_within_id = hashed_qid % DATASET_QUERY_NUM_UPPER_BOUND
    return f"{dataset_id}:{data_within_id}"

def unhash_did(hashed_did):
    dataset_id = hashed_did // DATASET_CAN_NUM_UPPER_BOUND
    data_within_id = hashed_did % DATASET_CAN_NUM_UPPER_BOUND
    return f"{dataset_id}:{data_within_id}"

def load_qrel(filename):
    qrel = {}
    qid_to_taskid = {}
    with open(filename, "r") as f:
        for line in f:
            query_id, _, doc_id, relevance_score, task_id = line.strip().split()
            if int(relevance_score) > 0:  # Assuming only positive relevance scores indicate relevant documents
                if query_id not in qrel:
                    qrel[query_id] = []
                qrel[query_id].append(doc_id)
                if query_id not in qid_to_taskid:
                    qid_to_taskid[query_id] = task_id
    print(f"Retriever: Loaded {len(qrel)} queries from {filename}")
    print(
        f"Retriever: Average number of relevant documents per query: {sum(len(v) for v in qrel.values()) / len(qrel):.2f}"
    )
    return qrel, qid_to_taskid

def compute_recall_at_k(relevant_docs, retrieved_indices, k):
    if not relevant_docs:
        return 0.0 # Return 0 if there are no relevant documents

    # Get the set of indices for the top k retrieved documents
    top_k_retrieved_indices_set = set(retrieved_indices[:k])

    # Convert the relevant documents to a set
    relevant_docs_set = set(relevant_docs)

    # Check if there is an intersection between relevant docs and top k retrieved docs
    # If there is, we return 1, indicating successful retrieval; otherwise, we return 0
    if relevant_docs_set.intersection(top_k_retrieved_indices_set):
        return 1.0
    else:
        return 0.0



In [4]:

class Args:
    def __init__(self):
        # Define the environment variables from the command
        _MODEL_ID = "./checkpoints/LamRA-Ret"
        _ORIGINAL_MODEL_ID = "Qwen/Qwen2-VL-7B-Instruct"
        _IMAGE_PATH_PREFIX = "/mnt/tidal-alsh01/dataset/mmeb/M-BEIR"

        # Arguments passed in the command line
        self.query_data_path: str = f"{_IMAGE_PATH_PREFIX}/query/test/mbeir_xhs_task7_test.jsonl"
        self.query_cand_pool_path: str = f"{_IMAGE_PATH_PREFIX}/cand_pool/local/mbeir_xhs_task7_cand_pool.jsonl"
        self.cand_pool_path: str = f"{_IMAGE_PATH_PREFIX}/cand_pool/local/mbeir_xhs_task7_cand_pool.jsonl"
        self.instructions_path: str = f"{_IMAGE_PATH_PREFIX}/instructions/query_instructions.tsv"
        self.qrels_path: str = f"{_IMAGE_PATH_PREFIX}/qrels/test/mbeir_xhs_task7_test_qrels.txt"
        self.original_model_id: str = _ORIGINAL_MODEL_ID
        self.image_path_prefix: str = _IMAGE_PATH_PREFIX
        self.model_id: str = _MODEL_ID

        # Argument with a default value from the argparse definition (not overridden in the command)
        self.model_max_length: int = 1024

# xhs 评估指标验证

In [84]:
args = Args()
from dataset.datasets_mbeir import QueryDataset, CandidateDataset
from PIL import Image
cand_dataset = CandidateDataset(
    query_data_path=args.query_data_path, 
    cand_pool_path=args.cand_pool_path,
    instructions_path=args.instructions_path,
    image_path_prefix=args.image_path_prefix
)
query_dataset = QueryDataset(
    query_data_path=args.query_data_path, 
    cand_pool_path=args.query_cand_pool_path,
    instructions_path=args.instructions_path,
    image_path_prefix=args.image_path_prefix
)


In [85]:
qrel, _ = load_qrel(args.qrels_path)
cand_pool = {d['did']:d for d in cand_dataset.cand_pool}

Retriever: Loaded 693 queries from /mnt/tidal-alsh01/dataset/mmeb/M-BEIR/qrels/test/mbeir_xhs_task7_test_qrels.txt
Retriever: Average number of relevant documents per query: 1.00


In [83]:

def show_some_query_info(query_idx):
    query, _ = query_dataset[query_idx]
    qimg = query[0]['content'][0]['image']
    # cands = cand_names_json[query_idx] # len 50
    poscand_did_list = qrel[f'10:{query_idx+1}']
    # print(poscand_did)
    # for i in (1,5,10,50):
    #     for poscand_did in poscand_did_list:
    #         if poscand_did in cands[:i]:
    #             print(f"right in recall@{i}")
    #             break
    imglist = [qimg]
    imglist.extend([ cand_pool[did]['img_path'] for did in poscand_did_list])
    show_group_imgs(imglist)
    return query, poscand_did_list


def show_some_cand(did):
    some_cand = cand_pool[did]
    return Image.open(some_cand['img_path'])

def show_group_imgs(image_paths, box_list=None, output_path=None):
    images = [Image.open(path).resize((224,224)) for path in image_paths]
    
    # 获取第一张图片的模式和大小
    mode = images[0].mode
    width, height = images[0].size
    
    # 检查所有图片是否模式一致
    for img in images:
        if img.mode != mode:
            raise ValueError("所有图片必须具有相同的模式")
    
    # 计算拼接后总宽度和高度
    total_width = sum(img.width for img in images)
    max_height = max(img.height for img in images)
    
    # 创建空白画布
    result = Image.new(mode, (total_width, max_height), (0, 0, 0, 0))
    
    # 拼接图片
    x_offset = 0
    for img in images:
        result.paste(img, (x_offset, 0))
        x_offset += img.width
    
    # 显示结果
    result.show()

def diff_topk_cands(ours, baseline, topk=5):
    strong, weak = [] , []
    for query_idx in range(len(query_dataset)):
        # for t in qrel[f'10:{query_idx+1}']:
        t = np.random.choice(qrel[f'10:{query_idx+1}'])
        if t in ours[query_idx][:topk] and (t not in baseline[query_idx][:topk]):
            strong.append(query_idx)
        elif t not in ours[query_idx][:topk] and (t in baseline[query_idx][:topk]):
            weak.append(query_idx)
    print(f"strong:{len(strong)}, weak:{len(weak)}")
    return strong, weak


In [83]:
# 展示poscand
# print(poscand_did_list)
# show_some_cand(poscand_did_list[0])

# 第一组实验，发现效果变差了  note-1k-1top-1pos-filter

In [85]:
rootdir = "LamRA_Ret_eval_results/note-1k-1top-1pos-filter"

with open("mbeir_xhs_task7_test_qwen2_5-vl-7b_LEMUIR_tune_bs120_cand_names.json", "r") as f:
    ourscand_names_json = json.load(f)

with open("mbeir_xhs_task7_test_qwen2_5-vl-7b_LEMUIR_tune_nodam_cand_names.json", "r") as f:
    baselinecand_names_json = json.load(f)    

In [86]:
strong, weak = diff_topk_cands(ourscand_names_json, baselinecand_names_json)

strong:1, weak:209


In [ ]:
# 展示top 5 的cands
# show_some_cand(cands[4])
# 如果是ours
query_idx = weak[1]
show_some_query_info(query_idx)
print("our results")

ourcands = ourscand_names_json[query_idx] # len 50
show_group_imgs([cand_pool[ourcands[i]]['img_path'] for i in range(10)])
print("baseline results")
baselinecands = baselinecand_names_json[query_idx]
show_group_imgs([cand_pool[baselinecands[i]]['img_path'] for i in range(10)])

# 第二组，原来recall@10有所提高的

In [33]:
rootdir = "LamRA_Ret_eval_results/note-693-10top-10cand-nofilter/"

with open(rootdir+"mbeir_xhs_task7_test_qwen2_5-vl-7b_LEMUIR_tune_bs120_cand_names.json", "r") as f:
    ourscand_names_json = json.load(f)

with open(rootdir+"mbeir_xhs_task7_test_qwen2_5-vl-7b_LEMUIR_tune_nodam_cand_names.json", "r") as f:
    baselinecand_names_json = json.load(f)    

In [74]:
strong, weak = diff_topk_cands(ourscand_names_json, baselinecand_names_json, topk=10)

strong:67, weak:64


In [ ]:
# 展示top 5 的cands
# show_some_cand(cands[4])
# 如果是ours
query_idx = strong[23]
query, pos_cand = show_some_query_info(query_idx)
print("our results")

ourcands = ourscand_names_json[query_idx] # len 50
show_group_imgs([cand_pool[ourcands[i]]['img_path'] for i in range(10)])
print("baseline results")
baselinecands = baselinecand_names_json[query_idx]
show_group_imgs([cand_pool[baselinecands[i]]['img_path'] for i in range(10)])

# 第3组， 最初的评估集

In [98]:
rootdir = "LamRA_Ret_eval_results/original-bug/"

with open(rootdir+"mbeir_xhs_task7_test_qwen2_5-vl-7b_LEMUIR_tune_bs120_cand_names.json", "r") as f:
    ourscand_names_json = json.load(f)

with open(rootdir+"mbeir_xhs_task7_test_qwen2_5-vl-7b_LEMUIR_tune_nodam_cand_names.json", "r") as f:
    baselinecand_names_json = json.load(f)    

In [99]:
strong, weak = diff_topk_cands(ourscand_names_json, baselinecand_names_json, topk=5)

strong:37, weak:31


In [ ]:
# 展示top 5 的cands
# show_some_cand(cands[4])
# 如果是ours
query_idx = strong[14]
query, pos_cand = show_some_query_info(query_idx)
print("our results")

ourcands = ourscand_names_json[query_idx] # len 50
show_group_imgs([cand_pool[ourcands[i]]['img_path'] for i in range(10)])
print("baseline results")
baselinecands = baselinecand_names_json[query_idx]
show_group_imgs([cand_pool[baselinecands[i]]['img_path'] for i in range(10)])